In [ ]:
import torch
import numpy as np
import torchvision
import torch.nn as nn
from torchvision.datasets import mnist

In [ ]:
transform = torchvision.transforms.Compose([
    # CIFAR-10 images are 3 channels, we might need to adjust the VAE input dimension later
    torchvision.transforms.Grayscale(num_output_channels=1), # Keep as 3 channels for now
    torchvision.transforms.ToTensor(),
    # Removed normalization to keep data in [0, 1] range for consistency with Noising function and loss
    # torchvision.transforms.Normalize((0.5,), (0.5,))
])

# Load the full CIFAR-10 training data
full_train_data = torchvision.datasets.CIFAR10(root="./data", train=True, transform=transform, download=True)

# Filter the dataset to include only images with a specific label (e.g., label 3 for cat)
# You can change the label number to train on a different class
target_label = 3
seven_indices = [i for i, (image, label) in enumerate(full_train_data) if label == target_label]
train_data_seven = torch.utils.data.Subset(full_train_data, seven_indices)

# Use the filtered dataset for training
train_data = train_data_seven

In [ ]:
def Noising(or_images, num_steps=10, theta = 4):
    noisy_image = or_images
    noise_tree = []
    for i in range(num_steps):
      noise_tree.append(noisy_image)
      noisy_image = theta * noisy_image * (1 - noisy_image)


    noise_tree.append(noisy_image)

    noise_added = noisy_image - or_images

    noise_tree = torch.stack(noise_tree)
    noise_tree = torch.flip(noise_tree, dims=[0])

    return noisy_image, noise_tree, noise_added

# xT → ... → x2 → x1 → x0
# shape = [t , 32, 32]

if __name__ == "__main__":
  rand_img = torch.rand(32, 32)
  noisy_img, noise_tree, noise_added = Noising(rand_img)
  print(noisy_img.shape)
  print(noise_tree.shape)
  print(noise_added.shape)

torch.Size([32, 32])
torch.Size([11, 32, 32])
torch.Size([32, 32])


In [ ]:
import torch

def Tensor2tree_tensor(x_t, num_steps=5, theta=4.0):
    """
    x_t: (B,C,H,W) or (H,W) or (C,H,W)
    returns: tree_tensor -> [steps+1 , nodes , B , C , H , W]
    """

    # -------- Ensure input in [B,C,H,W] --------
    if x_t.dim() == 2:
        x_t = x_t.unsqueeze(0).unsqueeze(0)    # (1,1,H,W)
    elif x_t.dim() == 3:
        x_t = x_t.unsqueeze(0)                 # (1,C,H,W)
    elif x_t.dim() != 4:
        raise ValueError("Input must be 2D/3D/4D image tensor")

    # ---------- Tree container ----------
    tree = []
    current = x_t.unsqueeze(0)                 # (1, B,C,H,W)
    tree.append(current)

    # ---------- Reverse expansion ----------
    for step in range(num_steps):

        disc = theta**2 - 4*theta*current
        disc = torch.clamp(disc, min=0)        # avoid imaginary numbers
        sqrt_disc = torch.sqrt(disc)

        # two logistic pre-images
        x1 = (-theta + sqrt_disc) / (-2*theta)
        x2 = (-theta - sqrt_disc) / (-2*theta)

        current = torch.cat([x1, x2], dim=0)   # → double node count
        tree.append(current)

    # final stacked format
    max_nodes = tree[-1].shape[0]

    # pad smaller levels to uniform size
    padded = []
    for level in tree:
        nodes = level.shape[0]
        if nodes < max_nodes:
            # repeat to match final shape
            repeat_factor = max_nodes // nodes
            level = level.repeat(repeat_factor,1,1,1,1)
        padded.append(level)

    return torch.stack(padded)                  # [steps+1 , 2^depth , B,C,H,W]

'''
Level-0:  1 node              → shape [1,B,C,H,W]
Level-1:  2 nodes             → shape [2,B,C,H,W]
Level-2:  4 nodes             → shape [4,B,C,H,W]
...
Level-n:  2^n nodes           → shape [2^n,B,C,H,W]
'''
if __name__ == "__main__":
  rand_img = torch.rand(32,1,32,32)
  tree = Tensor2tree_tensor(rand_img, num_steps=10)

  print(rand_img.shape)
  print(tree.shape, "[steps+1 , 2^depth , B,C,H,W]")

torch.Size([32, 1, 32, 32])
torch.Size([11, 1024, 32, 1, 32, 32]) [steps+1 , 2^depth , B,C,H,W]


In [ ]:
def theoretical_path(noise_tree):
    """
    noise_tree: [T+1, B, C, H, W]
    return    : [H*W, T+1]
    """

    # merge batch and channel → (T+1, B*C, H, W)
    noise_tree = noise_tree.flatten(1,2)  # [11, 32,1 → 32 , 32 ,32]

    # Take first sample or you can aggregate differently
    noise_tree = noise_tree[:,0,...]      # [T+1 , H , W]

    # reverse time so we get x0 → xT
    noise_tree = noise_tree.flip(0)

    T, H, W = noise_tree.shape

    # reshape to [T , H*W] and then transpose
    return noise_tree.reshape(T, H*W).T   # → [H*W , T]


if __name__ == "__main__":
  rand_img = torch.rand(32,1,32,32)
  noisy_image, noise_tree, _ = Noising(rand_img, num_steps=10)

  print(noise_tree.shape)        # ---> [11,32,1,32,32]
  result = theoretical_path(noise_tree)
  print(result.shape)            # ---> [1024 , 11]

torch.Size([11, 32, 1, 32, 32])
torch.Size([1024, 11])


In [ ]:
import torch

def tree_tensor_to_binary_graph(tree_tensor):
    """
    tree_tensor: [L , N , B , C , H , W]
                 where N = 2^level

    Returns:
        node_features : [Total_nodes , C , H , W]
        edge_index    : [2 , Total_edges]
        level_index   : list of node indices at each level
    """

    L, Nnodes_padded, B, C, H, W = tree_tensor.shape # Nnodes_padded is the max nodes (2^L-1)

    assert B == 1, "Binary graph supports B=1 currently"

    # ----- Collect nodes -----
    node_features = []
    level_index = []
    global_id = 0

    for level in range(L):
        num_nodes = 2**level  # Use the actual number of nodes for the current level
        level_nodes = tree_tensor[level, :num_nodes, 0]  # Extract only the unique nodes, dropping batch dim

        # Track indices
        ids = list(range(global_id, global_id + num_nodes))
        level_index.append(ids)

        global_id += num_nodes
        node_features.append(level_nodes)

    # stack all nodes
    node_features = torch.cat(node_features, dim=0)  # [Total_nodes, C, H, W]

    # ----- Create binary edges -----
    edges_src = []
    edges_dst = []

    for level in range(L - 1):

        parents  = level_index[level]
        children = level_index[level + 1]

        assert len(children) == 2 * len(parents), \
            f"Level {level} has {len(parents)} parents but {len(children)} children (expect 2×)"

        for i, p in enumerate(parents):
            c1 = children[2 * i]
            c2 = children[2 * i + 1]

            edges_src += [p, p]
            edges_dst += [c1, c2]

    edge_index = torch.tensor([edges_src, edges_dst], dtype=torch.long)

    return node_features, edge_index, level_index



# ---------- DEMO (using your Tensor2tree_tensor output) ----------
if __name__ == "__main__":
    rand_img = torch.rand(1,3,32,32) # Changed batch size from 32 to 1
    tree = Tensor2tree_tensor(rand_img, num_steps=5)

    print("Tree shape:", tree.shape)

    nodes, edges, levels = tree_tensor_to_binary_graph(tree)

    print("Nodes:", nodes.shape)
    print("Edges:", edges.shape)
    print("Levels:", levels)


Tree shape: torch.Size([6, 32, 1, 3, 32, 32])
Nodes: torch.Size([63, 3, 32, 32])
Edges: torch.Size([2, 62])
Levels: [[0], [1, 2], [3, 4, 5, 6], [7, 8, 9, 10, 11, 12, 13, 14], [15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30], [31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62]]


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleGNN(nn.Module):
    def __init__(self, C, hidden=64):
        super().__init__()
        self.encoder = nn.Conv2d(C, hidden, 3, padding=1)
        self.gcn1 = nn.Linear(hidden, hidden)
        self.gcn2 = nn.Linear(hidden, hidden)
        self.decoder = nn.Conv2d(hidden, C, 3, padding=1)

    def propagate(self, x, edge_index):
        src, dst = edge_index  # [2, E]
        agg = torch.zeros_like(x)
        agg.index_add_(0, dst, x[src])
        return agg

    def forward(self, node_features, edge_index):
        # node_features: [N, C, H, W]
        N, C, H, W = node_features.shape

        x = self.encoder(node_features)       # [N, hidden, H, W]
        x = x.mean(dim=[2,3])                 # [N, hidden] (flatten)

        x = F.relu(self.gcn1(self.propagate(x, edge_index)))
        x = F.relu(self.gcn2(self.propagate(x, edge_index)))

        # decode graph output back to image
        x = x.view(N, -1, 1, 1).expand(N, -1, H, W)
        out = self.decoder(x)

        return out

In [ ]:
from torch.utils.data import dataloader
num_epochs = 20
batch_size = 32

train_data = dataloader(train_data, )